# NEU-ESC Adaptive Interaction
GPU workflow with resumable 4 variants × 5 paired seeds (42-46).
Runs will save metrics to runs/adaptive_interaction and aggregate into reports/adaptive_interaction.

In [ ]:
!nvidia-smi
import torch, platform
print('Platform:', platform.platform())
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU accelerator is required for training runs!'

In [ ]:
# Clean up any previous clone to guarantee fresh code
!rm -rf /kaggle/working/NEU_ESC
!git clone https://github.com/AI-Peak/neu-esc-reproduction.git /kaggle/working/NEU_ESC
%cd /kaggle/working/NEU_ESC
!pip install -q 'transformers<5' sentencepiece
!python src_baseline/neu_esc_kaggle.py fetch --data-dir data/raw/kaggle
!python -m src_adaptive.data_audit data/NEU --out reports/adaptive_interaction/kaggle_data_audit.json

In [ ]:
from pathlib import Path
import json, subprocess

variants = ['baseline', 'independent', 'ungated', 'adaptive']
seeds = [42, 43, 44, 45, 46]
root = Path('runs/adaptive_interaction')
root.mkdir(parents=True, exist_ok=True)

for variant in variants:
    for seed in seeds:
        run_dir = root / f'{variant}-s{seed}'
        if (run_dir / 'metrics.json').exists():
            print(f'==> Skip completed: {run_dir}')
            continue
        print(f'\n========================================')
        print(f'Starting experiment: variant={variant}, seed={seed}')
        print(f'========================================\n')
        subprocess.run([
            'python', 'src_adaptive/run_experiment.py',
            '--variant', variant,
            '--seed', str(seed),
            '--data-dir', 'data/raw/kaggle',
            '--processed',
            '--revision', '86cd7fd4c148980922ac11a2cf5e257f2ba639e1',
            '--out-dir', str(root)
        ], check=True)

In [ ]:
# Aggregate metrics and run unit tests
!python src_adaptive/aggregate.py runs/adaptive_interaction --out reports/adaptive_interaction
!python src_adaptive/test_adaptive.py

# Copy results to /kaggle/working root for easy download
!cp -r reports/adaptive_interaction /kaggle/working/
!cp -r runs/adaptive_interaction /kaggle/working/
print('All runs completed successfully!')